# 3 — Preprocessing


## Imports


In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import wfdb
from scipy.signal import butter, sosfiltfilt


## Configuration


In [2]:
from pathlib import Path
# ใช้ project root เดียวกัน ไม่ต้อง import โค้ด pipeline จาก program
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'dataset').is_dir() and (p / 'model').is_dir() and (p / 'program').is_dir())
RAW = ROOT / 'dataset/raw/mit-bih-arrhythmia-database-1.0.0'
DATA = ROOT / 'dataset/processed'
ARTIFACTS = ROOT / 'model/artifacts'
REPORTS = ROOT / 'model/reports'
CLASSES = ['N', 'S', 'V', 'F', 'Q']
SYMBOLS = ['NLRej', 'AaJS', 'VE', 'F', '/fQ']
LABEL_MAP = {s: i for i, group in enumerate(SYMBOLS) for s in group}
SEED = 42
PREPROCESSING = {'version': 1, 'fs': 360, 'left': 90, 'right': 162,
                 'band': [0.5, 40], 'order': 4, 'leads': ['MLII', 'V5']}

## Artifact helpers


In [3]:
def read_json(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'ไม่พบ {path}; กรุณารัน notebook ขั้นก่อนหน้าก่อน')
    return json.loads(path.read_text(encoding='utf-8'))

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def new_stage(marker):
    # ห้ามเขียนทับผลเดิมเงียบ ๆ เพราะอาจทำให้โมเดลอ้างอิงคนละ split
    if Path(marker).exists():
        raise FileExistsError(f'มีผลขั้นนี้แล้ว: {marker}; โหลดผลเดิมแทนการรันซ้ำ')
    if (ARTIFACTS / 'selection.json').exists():
        raise RuntimeError('ล็อกโมเดลแล้ว ไม่อนุญาตให้เตรียมข้อมูลหรือ tuning ใหม่ใน experiment เดิม')

## Patient leakage checks


In [4]:
def validate_split(beats):
    if set(beats.split) != {'train', 'validation', 'test'}:
        raise ValueError('ต้องมีเพียง train/validation/test')
    if beats.groupby('patient_group').split.nunique().max() != 1:
        raise ValueError('พบ patient leakage')
    for name, part in beats.groupby('split'):
        if set(part.class_id) != set(range(5)):
            raise ValueError(f'{name} มีคลาสไม่ครบ 5; หยุดก่อนฝึก ห้ามแก้ split จากคะแนนโมเดล')
    paired = beats[beats.record_id.isin(['201', '202'])]
    if not paired.empty and (paired.patient_group.nunique() != 1 or paired.split.nunique() != 1):
        raise ValueError('201/202 ต้องเป็นผู้ป่วยกลุ่มเดียวและชุดเดียวกัน')

## Lead selection and record reading


In [5]:
def choose_lead(names):
    for name in PREPROCESSING['leads']:
        if name in names:
            return names.index(name), name
    raise ValueError(f'ไม่มี lead MLII หรือ V5: {names}')

def read_record(path):
    record = wfdb.rdrecord(str(path))
    if record.fs != PREPROCESSING['fs']:
        raise ValueError(f'ต้องการ 360 Hz แต่พบ {record.fs}')
    index, lead = choose_lead(record.sig_name)
    signal = np.asarray(record.p_signal[:, index], dtype=np.float64)
    if len(signal) != record.sig_len or not np.isfinite(signal).all():
        # ไม่ filter ข้าม NaN เพราะจะกระจายความเสียหายไปทั้ง record
        raise ValueError(f'{path}: signal ไม่ครบหรือมี NaN/Inf ต้องตรวจสอบก่อน')
    return signal, lead

## Filtering and segmentation


In [6]:
def segment(signal, samples):
    samples = np.asarray(samples, dtype=np.int64)
    if samples.ndim != 1 or np.any(np.diff(samples) <= 0):
        raise ValueError('ตำแหน่ง heartbeat ต้องเรียงและไม่ซ้ำ')
    if np.any(samples < 0) or np.any(samples >= len(signal)):
        raise ValueError('ตำแหน่ง annotation อยู่นอก signal')
    sos = butter(4, [0.5, 40], btype='bandpass', fs=360, output='sos')
    filtered = sosfiltfilt(sos, signal)
    valid = (samples >= 90) & (samples + 162 <= len(signal))
    waves = np.array([filtered[r - 90:r + 162] for r in samples[valid]], dtype=np.float32)
    if not len(waves):
        raise ValueError('ไม่มี heartbeat ที่ตัดหน้าต่าง 252 samples ได้')
    if waves.shape[1] != 252 or not np.isfinite(waves).all():
        raise ValueError('waveform หลัง filter ใช้งานไม่ได้')
    return waves, valid

## RR context


In [7]:
def rr_features(samples):
    # คำนวณบนลำดับ beats ทั้ง record ก่อนตัด boundary beats ทิ้ง
    intervals = np.diff(samples) / 360.0
    previous = np.r_[np.nan, intervals]
    following = np.r_[intervals, np.nan]
    trailing = pd.Series(previous).rolling(10, min_periods=1).mean().to_numpy()
    return np.column_stack([previous, following, previous / following, trailing])

## Verify input artifacts


In [8]:
raw = RAW
new_stage(DATA / 'preprocessing.json')
split = read_json(DATA / 'split.json')
audit_info = read_json(DATA / 'audit.json')
if sha256(DATA / 'audit.json') != split['audit_hash']:
    raise ValueError('audit เปลี่ยนหลัง split')
for name, expected in audit_info['files'].items():
    if sha256(raw / name) != expected:
        raise ValueError(f'raw file เปลี่ยน: {name}')
inventory = pd.read_parquet(DATA / 'split_inventory.parquet')

## Process each record


In [9]:
chunks, waves, rrs, exclusions = [], [], [], []
for record_id, part in inventory.groupby('record_id', sort=True):
    part = part.sort_values('r_sample').reset_index(drop=True)
    signal, _ = read_record(raw / record_id)
    samples = part.r_sample.to_numpy()
    wave, valid = segment(signal, samples)
    rr = rr_features(samples)[valid]
    exclusions.extend(part.loc[~valid, ['beat_id', 'record_id']].assign(reason='window_boundary').to_dict('records'))
    chunks.append(part.loc[valid].copy())
    waves.append(wave)
    rrs.append(rr)

## Fit normalization on Train


In [10]:
meta = pd.concat(chunks, ignore_index=True)
wave = np.concatenate(waves)
rr = np.concatenate(rrs)
validate_split(meta)
# Fit เฉพาะ Train; validation/test ได้ transform ด้วยค่าเดียวกัน
train_wave = wave[meta.split.eq('train').to_numpy()]
mean, std = float(train_wave.mean(dtype=np.float64)), float(train_wave.std(dtype=np.float64))
if std <= 1e-12:
    raise ValueError('Train waveform ไม่มี variance')
state = {**PREPROCESSING, 'mean': mean, 'std': std, 'split_hash': split['split_hash'],
         'audit_hash': split['audit_hash'], 'classes': CLASSES, 'label_map': LABEL_MAP}

## Save each partition


In [11]:
for name in ['train', 'validation', 'test']:
    mask = meta.split.eq(name).to_numpy()
    folder = DATA / name
    folder.mkdir(parents=True, exist_ok=True)
    selected = meta.loc[mask].copy().reset_index(drop=True)
    selected['waveform_index'] = np.arange(len(selected))
    selected.to_parquet(folder / 'metadata.parquet', index=False)
    np.save(folder / 'waveforms.npy', ((wave[mask] - mean) / std).astype(np.float32))
    np.save(folder / 'rr.npy', rr[mask])
pd.DataFrame(exclusions, columns=['beat_id', 'record_id', 'reason']).to_csv(DATA / 'excluded_beats.csv', index=False)
write_json(DATA / 'preprocessing.json', state)
meta.groupby(['split', 'class_id']).size().unstack(fill_value=0)

class_id,0,1,2,3,4
split,,,,,
test,8484,279,772,25,2062
train,72395,2254,5799,755,4176
validation,9715,248,664,22,1802
